# Lab MQTT — 03. Message Analysis

Loads the file produced by `02_mqtt_collector.py` (a list of records, one MQTT message per
row) and reproduces, on our own dataset, the main analyses of *MQTT in the Wild*:

1. **Payload type** distribution (string / json / numeric / bool / unknown) → paper Fig. 9
2. **QoS** distribution (0 / 1 / 2)
3. % of **retained** messages per broker → paper Fig. 10c
4. **Payload length CDF** → paper Fig. 10a
5. **Topic depth and length CDFs** → paper Fig. 8
6. Observed **throughput** per broker (messages/min)
7. **Top topics** received per broker

In [ ]:
import gzip, pickle
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

INPUT = Path("./outputs/captured_messages.pkl.gz")  # change if needed
OUT_DIR = Path("./outputs"); OUT_DIR.mkdir(exist_ok=True)

with gzip.open(INPUT, "rb") as fh:
    records = pickle.load(fh)

df = pd.DataFrame(records)
df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
print("Total messages:", len(df))
print("Brokers:", df['broker'].unique())
print("Period:", df['datetime'].min(), '→', df['datetime'].max())
df.head()

## 1. Payload type distribution (paper Fig. 9)

In [ ]:
ORDER = ['string', 'json', 'numeric', 'unknown', 'bool']
vc = df['payload_type'].value_counts(normalize=True).reindex(ORDER, fill_value=0) * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(vc.index, vc.values, edgecolor='black')
for b, v in zip(bars, vc.values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.5, f"{v:.1f}%", ha='center')
ax.set_ylabel('Percentage of Messages')
ax.set_xlabel('Payload Type')
ax.set_title('Payload type distribution')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_payload_types.png", dpi=140)
plt.show()

## 2. QoS distribution

**Important:** we subscribe at QoS 2, so the `qos` field of the received message shows the
**publisher's original QoS** (MQTT delivers at the minimum between published and subscribed
QoS, and since we are at 2 the value is not degraded).

In [ ]:
qos_dist = df['qos'].value_counts(normalize=True).sort_index() * 100
print(qos_dist)

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar([f'QoS {q}' for q in qos_dist.index], qos_dist.values, edgecolor='black')
for b, v in zip(bars, qos_dist.values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.5, f"{v:.1f}%", ha='center')
ax.set_ylabel('Percentage of Messages')
ax.set_title('QoS distribution observed')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_qos.png", dpi=140)
plt.show()

## 3. Retain flag — per broker

In [ ]:
retain_per_broker = df.groupby('broker')['retain'].mean() * 100
retain_per_broker = retain_per_broker.sort_values(ascending=False)
print(retain_per_broker)

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(retain_per_broker.index, retain_per_broker.values, edgecolor='black')
for b, v in zip(bars, retain_per_broker.values):
    ax.text(b.get_x() + b.get_width()/2, v + 0.5, f"{v:.1f}%", ha='center')
ax.set_ylabel('% retained messages')
ax.set_title('Retain flag — per broker')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_retain.png", dpi=140)
plt.show()

## 4. Payload length CDF (paper Fig. 10a)

In [ ]:
def cdf(arr):
    arr = np.sort(np.asarray(arr))
    y = np.arange(1, len(arr)+1) / len(arr)
    return arr, y

fig, ax = plt.subplots(figsize=(7, 4))
for name, sub in df.groupby('broker'):
    x, y = cdf(sub['payload_length'].values)
    ax.step(np.maximum(x, 1), y, where='post', label=name)
x, y = cdf(df['payload_length'].values)
ax.step(np.maximum(x, 1), y, where='post', linestyle='--', color='black', label='Total')
ax.set_xscale('log')
ax.set_xlabel('Payload length (bytes)')
ax.set_ylabel('CDF (% of messages)')
ax.set_title('Payload length CDF')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_payload_length_cdf.png", dpi=140)
plt.show()

for q in (0.5, 0.9, 0.99):
    print(f"P{int(q*100)} payload length: {int(np.quantile(df['payload_length'], q))} bytes")

## 5. Topic depth and length CDF (paper Fig. 8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, sub in df.groupby('broker'):
    x, y = cdf(sub['topic_depth'].values)
    axes[0].step(x, y, where='post', label=name)
    x, y = cdf(sub['topic_length'].values)
    axes[1].step(x, y, where='post', label=name)
axes[0].set_xscale('log'); axes[0].set_xlabel('Topic depth'); axes[0].set_ylabel('CDF')
axes[0].grid(True, alpha=0.3); axes[0].legend()
axes[1].set_xscale('log'); axes[1].set_xlabel('Topic length (bytes)'); axes[1].set_ylabel('CDF')
axes[1].grid(True, alpha=0.3); axes[1].legend()
axes[0].set_title('Topic depth CDF')
axes[1].set_title('Topic length CDF')
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_topic_cdf.png", dpi=140)
plt.show()

## 6. Throughput per broker (msg/minute)

In [ ]:
df['minute'] = df['datetime'].dt.floor('min')
rate = df.groupby(['minute', 'broker']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
rate.plot(ax=ax)
ax.set_ylabel('Messages / minute')
ax.set_xlabel('Time')
ax.set_title('Per-broker throughput over time')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_throughput.png", dpi=140)
plt.show()

print("Mean msg/min per broker:")
print(rate.mean().round(1))

## 7. Top topics per broker

In [ ]:
for name, sub in df.groupby('broker'):
    print(f"\n=== {name} — top 10 topics ===")
    print(sub['topic'].value_counts().head(10))

## 8. Summary table for the presentation

In [ ]:
summary = df.groupby('broker').agg(
    n_messages       = ('topic', 'size'),
    n_unique_topics  = ('topic', 'nunique'),
    pct_retained     = ('retain', lambda s: 100*s.mean()),
    pct_qos0         = ('qos', lambda s: 100*(s == 0).mean()),
    pct_qos1         = ('qos', lambda s: 100*(s == 1).mean()),
    pct_qos2         = ('qos', lambda s: 100*(s == 2).mean()),
    median_payload_B = ('payload_length', 'median'),
    median_topic_dep = ('topic_depth', 'median'),
).round(2)
summary.to_csv(OUT_DIR / "per_broker_summary.csv")
summary